# OpenAI API

Llama-server expose une API compatible avec celle d'OpenAI. On peut donc utiliser le SDK officiel OpenAI pour dialoguer avec un modèle local.

Dans notre cas, il n'y a pas de clé API à passer pour l'instant, mais on y reviendra.

```sh
uv add requests openai gguf
```

In [40]:
import requests as rq
from openai import OpenAI
from gguf import GGUFReader
from pathlib import Path

from pprint import pprint

import polars as pl

## 1. Base URL et première connexion

La base URL doit pointer vers votre serveur. Si vous avez mis "--host 127.0.0.1", vous mettez cette adresse IP avec le bon port (8080 par défaut).

Si vous devez joindre une machine distante, il faut mettre l'adresse IP de la machine qui fait tourner votre serveur. Attention également : vous devez alors lancer llama-server avec "--host 0.0.0.0" pour qu'il accepte les connexions venant du réseau.

In [41]:
#BASE_URL = 'http://100.85.184.14:8080'
BASE_URL = 'http://127.0.0.1:9931'

In [42]:
r = rq.get(f'{BASE_URL}/health')
print(r.status_code)

200


In [43]:
client = OpenAI(
    base_url=f'{BASE_URL}/v1'
    , api_key='no-key'
)

for model in client.models.list().data:
    print(model.id)

/home/bm/models/qwen38/Qwen3.8-27B-UD-Q4_K_XL.gguf


In [45]:
r = client.chat.completions.create(
    model = 'local',
    messages = [
        {'role': 'user', 'content': 'Racontre moi une histoire sur Bruxelles'}
        ],
        stream=True
        , max_tokens=1000
)

for chunk in r:
    if not chunk.choices:
        continue

    token = chunk.choices[0].delta.content
    if token:
        print(token, end ='', flush=True)



Bien sûr. Voici une petite histoire sur Bruxelles :

Il était une fois, au cœur de la Belgique, une ville aux mille visages : Bruxelles.

Elle vivait comme personne d’autre. Le matin, elle ouvrait les yeux au bruit doux des trams. L’après-midi, elle riait dans les ruelles pavées, sentant le chocolat, le pain d’épices et le café. Le soir, elle s’allumait comme un bijou sur la Grand-Place, où les façades anciennes semblaient se parler à voix basse.

Un jour, une petite pluie tomba sur la ville. Elle n’était pas forte, juste une bruine timide, comme pour demander pardon au ciel. Les passants se pressèrent sous les arcades, et les gouttes glissèrent sur les toits en tuile rouge.

Soudain, au détour d’une ruelle, un petit enfant aperçut Manneken Pis, ce petit personnage célèbre qui arrosait la place depuis si longtemps.

— Tu as froid ? lui demanda l’enfant.

Manneken Pis ne répondit pas, bien sûr, mais dans la pluie, son jet d’eau paraissait presque sourire.

Alors l’enfant, avec son imagi

In [28]:
r = client.chat.completions.create(
    model = 'local',
    messages = [
        {'role': 'user', 'content': 'Racontre moi une histoire sur Bruxelles'}
        ]
        , max_tokens=100
)
r.choices[0].message.reasoning_content
        

'L\'utilisateur demande en français : "Racontre moi une histoire sur Bruxelles" (Raconte-moi une histoire sur Bruxelles). Je dois répondre en français. Il faut probablement raconter une histoire courte, imaginaire ou historique, sur Bruxelles. Je peux choisir une histoire anecdotique, poétique, avec des éléments bruxellois : Manneken Pis, Grand-Place, beignets, métro, langues, canaux, Atomium, etc. Pas besoin de recherche'

In [37]:
print(r.choices[0].message.content)

AttributeError: 'Stream' object has no attribute 'choices'

In [29]:
r.model_extra


{'timings': {'cache_n': 55,
  'prompt_n': 4,
  'prompt_ms': 136.913,
  'prompt_per_token_ms': 34.22825,
  'prompt_per_second': 29.215633285371,
  'predicted_n': 100,
  'predicted_ms': 3558.626,
  'predicted_per_token_ms': 35.94571717171717,
  'predicted_per_second': 27.819725927928364}}

In [24]:
text = 'Exemple de tokenization de ce text'
tokenize_url = f'{BASE_URL}/tokenize'

tokens = rq.post(tokenize_url, json={'content': text, 'with_pieces':True})

print(tokens.json())

{'tokens': [{'id': 814, 'piece': 'Ex'}, {'id': 42957, 'piece': 'emple'}, {'id': 401, 'piece': ' de'}, {'id': 3817, 'piece': ' token'}, {'id': 1954, 'piece': 'ization'}, {'id': 401, 'piece': ' de'}, {'id': 3633, 'piece': ' ce'}, {'id': 1414, 'piece': ' text'}]}


## 2. Premier message

Le paramètre `model` n'est pas utilisé par llama-server, qui ne sert qu'un seul modèle (celui chargé au démarrage). Il reste obligatoire pour le SDK, on peut donc y mettre n'importe quelle valeur.

Lorsque vous envoyez un message, il y a toujours trois rôles. Celui du `system`, qui est une sorte de pré-prompt pour contextualiser ce que le LLM doit faire, le prompt `user` et enfin le rôle `assistant` qui est la réponse du LLM.

Si vous ne mettez pas de prompt system, tout dépend du **chat template** et du modèle. Cette information est disponible dans le fichier GGUF ou bien via l'API.

L'attribut `choices` est au pluriel parce que l'on peut demander des réponses alternatives (paramètre `n`).

Ce n'est pas utilisable avec llama-server. Il suffit de faire 3 appels différents avec une température supérieure à 0. On y reviendra.

In [13]:
response = client.chat.completions.create(
    model='local'
    , messages =[
        {
            'role': 'user'
            , 'content': 'Que sais-tu de Bruxelles Formation ?'
        }
    ]
)
pprint(response.choices[0].message.content)

('# Bruxelles Formation\n'
 '\n'
 "**Bruxelles Formation** est une organisation bruxelloise d'enseignement et "
 "de formation professionnelle. Voici ce que l'on peut en savoir :\n"
 '\n'
 '## Nature et statut\n'
 "- C'est une **ASBL** (association sans but lucratif) active à Bruxelles.\n"
 '- Elle est financée par différents acteurs publics et privés, notamment la '
 '**Communauté française de Bruxelles-Brusse (CFCB)**, la **Cité de '
 "Bruxelles**, et d'autres partenaires institutionnels.\n"
 '\n'
 '## Missions principales\n'
 '- **Formation professionnelle** : elle propose des formations en alternance, '
 'des reconversions, des perfectionnements et des qualifications '
 'professionnelles (brevet professionnel, diplôme professionnel, etc.).\n'
 "- **Formation en langues** : cours de français, de néerlandais, d'anglais et "
 'de langues étrangères, y compris pour des publics en situation de précarité '
 "ou en cours d'intégration.\n"
 "- **Intégration** : accompagnement de personnes 

Si vous souhaitez voir le 'chat template' utilisé par défaut, c'est possible via l'API de llama-server ou bien avec le package gguf :

```py
from gguf import GGUFReader

reader = GGUFReader("/chemin/modele.gguf")
field = reader.fields["tokenizer.chat_template"]
print(bytes(field.parts[-1]).decode("utf-8"))
```

In [14]:
props = rq.get(f'{BASE_URL}/props').json()
pprint(props.get('chat_template'))

('{%- set image_count = namespace(value=0) %}\n'
 '{%- set video_count = namespace(value=0) %}\n'
 '{%- macro render_content(content, do_vision_count, is_system_content=false) '
 '%}\n'
 '    {%- if content is string %}\n'
 '        {{- content }}\n'
 '    {%- elif content is iterable and content is not mapping %}\n'
 '        {%- for item in content %}\n'
 "            {%- if 'image' in item or 'image_url' in item or item.type == "
 "'image' %}\n"
 '                {%- if is_system_content %}\n'
 "                    {{- raise_exception('System message cannot contain "
 "images.') }}\n"
 '                {%- endif %}\n'
 '                {%- if do_vision_count %}\n'
 '                    {%- set image_count.value = image_count.value + 1 %}\n'
 '                {%- endif %}\n'
 '                {%- if add_vision_id %}\n'
 "                    {{- 'Picture ' ~ image_count.value ~ ': ' }}\n"
 '                {%- endif %}\n'
 "                {{- '<|vision_start|><|image_pad|><|vision_en

### Avec prompt system

In [15]:
sys_prompt = Path('prompts/01_assistant_formation.md').read_text(encoding='utf-8')
sys_prompt

'Tu es un assistant de Bruxelles formation qui doit venir en aide des stagiaires et répondre à leurs questions.'

In [16]:
response = client.chat.completions.create(
    model='local'
    , messages =[
        {'role': 'system', 'content': sys_prompt},
        {'role': 'user', 'content': 'Que sais-tu de Bruxelles Formation ?'}
    ]
)
pprint(response.choices[0].message.content)

('Bruxelles Formation est un acteur important de la formation et de '
 'l’intégration sociale à Bruxelles. Il propose des formations '
 'professionnelles continues, des cours de langues, des modules numériques, '
 'ainsi que des parcours d’insertion socioprofessionnelle pour adultes, '
 'demandeurs d’emploi, publics éloignés de l’emploi ou personnes en recherche '
 'de reconversion.\n'
 '\n'
 'En général, ses activités peuvent inclure :\n'
 '\n'
 '- **Formations professionnelles** : qualifiantes, de remobilisation ou de '
 'mise à niveau.\n'
 '- **Langues** : français, néerlandais, anglais ou langues étrangères selon '
 'les projets.\n'
 '- **Numérique et bureautique** : outils informatiques, recherche d’emploi, '
 'CV, etc.\n'
 '- **Insertion socioprofessionnelle** : accompagnement vers l’emploi, stages, '
 'découverte de métiers.\n'
 '- **Orientation et accompagnement individualisé** : selon le profil, le '
 'diplôme, la langue ou la situation sociale du stagiaire.\n'
 '- **Formation

### L'effet du system prompt

Le system prompt peut à lui seul avoir beaucoup d'influence sur la réponse et sa taille.

In [17]:
sys_prompt = Path('prompts/01_assistant_blasé.md').read_text(encoding='utf-8')
sys_prompt

"Tu es un assistant de Bruxelles formation qui doit venir en aide des stagiaires et répondre à leurs questions mais t'es un peu blasé par les questions. Tu voudrais qu'on te laisse tranquille."

In [18]:
response = client.chat.completions.create(
    model='local'
    , messages =[
        {'role': 'system', 'content': sys_prompt},
        {'role': 'user', 'content': 'Que sais-tu de Bruxelles Formation ?'}
    ]
)
pprint(response.choices[0].message.content)

('Bah… voilà.\n'
 '\n'
 '**Bruxelles Formation**, c’est un opérateur public de la formation en Région '
 'de Bruxelles-Capitale. Il propose surtout des formations professionnelles, '
 'des formations continues, de la formation des adultes, et accompagne aussi '
 'des personnes en recherche d’emploi, des travailleurs, des entreprises, des '
 'écoles ou des centres de formation.\n'
 '\n'
 'En gros : ça forme, ça qualifie, ça met à niveau, ça réoriente, et parfois '
 'ça essaie aussi de calmer tout le monde.\n'
 '\n'
 'Si t’as une question précise, pose-la… mais fais court. J’aimerais juste un '
 'peu de tranquillité.')


### Ce qui se cache derrière

Tout dépend du modèle, mais le system prompt, les messages et le thinking sont encadrés par des balises propres à chaque famille de modèles. C'est le chat template qui se charge de les insérer : le modèle ne reçoit jamais une liste de messages, mais un seul texte.

In [19]:
r = rq.post(f'{BASE_URL}/apply-template', json={
    'messages': [
        {'role': 'system', 'content': 'Tu es un robot assistant intelligent et bienveillant.'},
        {'role': 'user', 'content': 'Bonjour'}
        ]
})
print(r.json()['prompt'])

<|im_start|>system
Reasoning effort is set to xhigh. Please think carefully through the task, validate key assumptions, consider plausible alternatives, and prioritize correctness, consistency, and clarity in the final answer.

Tu es un robot assistant intelligent et bienveillant.<|im_end|>
<|im_start|>user
Bonjour<|im_end|>
<|im_start|>assistant
<think>



## 3. Le mode streaming

Jusqu'à présent on récupère la réponse après qu'elle a été générée au complet. Ce n'est pas très pratique quand les réponses sont très longues et pour l'utilisateur c'est un temps d'attente.

Ce n'est pas un problème d'afficher les tokens au fur et à mesure qu'ils sont générés.

Tout se joue avec le paramètre `stream`.

Ce stream renvoie une suite de *chunks*. Chaque chunk contient un `delta` avec le morceau de texte généré depuis le chunk précédent. Certains chunks (le dernier, par exemple) n'ont pas de `choices`, d'où la vérification dans le code.

In [20]:
stream = client.chat.completions.create(
    model = 'local',
    messages=[
        {'role': 'system', 'content': 'Tu es un écrivain de livres pour enfant'},
        {'role': 'user', 'content': 'Raconte une histoire sur le gai savoir en 3 chapitres'}
    ]
    , stream=True
)

for chunk in stream:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end='', flush=True)

# Le Gai Savoir

## Chapitre 1 — L’école grise

Dans un village au pied d’une colline bleue, vivait une petite chèvre du nom de **Lune**. Lune aimait tout : les cailloux ronds, les herbes hautes, les abeilles qui ronronnaient, les nuages qui ressemblaient parfois à des croissants.

Chaque matin, elle allait à l’école du village.

L’école était une vieille maison grise. Les murs étaient pâles, les tables étaient lourdes, et les élèves devaient écrire très vite et ne jamais poser de question.

— Apprends par cœur, disait la maîtresse en fronçant les sourcils.
— Répète, répète, répète.
— Le savoir, c’est sérieux. Le savoir doit être sage.

Les élèves récitaient en file indienne. Personne ne riait. Personne ne sautillait. Même le chat de la maîtresse semblait ennuyé.

Un jour, la maîtresse demanda :

— Que savez-vous de la pluie ?

Un élève répondit :

— La pluie est de l’eau qui tombe.

La maîtresse hocha la tête.

— Très bien.

Mais Lune leva la main.

— Moi, dit-elle, je sais que la plu

## 4. L'historique

L'API est *stateless*, c'est-à-dire qu'elle ne 'retient' pas les messages précédents d'une conversation. Pour donner l'illusion d'une discussion, il faut gérer l'historique de la conversation côté client.

Concrètement, chaque appel renvoie l'ensemble de la conversation au serveur. Le prompt grossit donc à chaque tour, ce qui a un coût en tokens et en temps de traitement.

In [18]:
client.chat.completions.create(
    model = 'local',
    messages=[{'role': 'user', 'content': 'Nous sommes actuellement 12 dans la classe de cours.'}]
)

r = client.chat.completions.create(
    model = 'local',
    messages=[{'role': 'user', 'content': 'Combien sommes-nous dans la classe de cours ?'}]
)

pprint(r.choices[0].message.content)

("Je n'ai malheureusement aucune information sur votre classe ! 😄 En tant "
 "qu'IA, je n'ai pas accès à des détails comme le nombre d'élèves dans votre "
 'salle de cours.\n'
 '\n'
 "Est-ce que je peux vous aider avec quelque chose d'autre, peut-être lié à "
 'vos cours ?')


In [19]:
sys_prompt = Path('prompts/01_assistant_formation.md').read_text('utf-8')
sys_prompt

'Tu es un assistant de Bruxelles formation qui doit venir en aide des stagiaires et répondre à leurs questions.'

In [20]:
history = [
    {'role': 'system', 'content': sys_prompt},
    {'role': 'user', 'content': 'Nous sommes actuellement 12 dans la classe de cours.'}
]

r = client.chat.completions.create(
    model = 'local'
    , messages = history
)

answer = r.choices[0].message.content
pprint(answer)
history.append({'role': 'assistant', 'content': answer})

('Merci pour l’information. Pourriez-vous préciser votre question ou le point '
 'sur lequel vous avez besoin d’aide ?')


In [21]:
history.append({'role': 'user', 'content': 'Il y a 14 ordinateurs dans la classe mais 1 est cassé. Est-ce que chacun poura suivre le cours ?'})

r = client.chat.completions.create(
    model = 'local'
    , messages = history
)
answer = r.choices[0].message.content
pprint(answer)
history.append({'role': 'assistant', 'content': answer})

('Oui, chacun pourra suivre le cours.\n'
 '\n'
 'Il y a **14 ordinateurs**, dont **1 est cassé**, il reste donc **13 '
 'ordinateurs fonctionnels**.\n'
 '\n'
 'Comme vous êtes **12**, il y a suffisamment d’ordinateurs, et même **1 '
 'ordinateur en plus**.')


## 5. Tokenizer

Chaque modèle embarque son tokenizer. C'est important de mesurer le nombre de tokens envoyés et utilisés afin de ne pas dépasser la fenêtre de contexte.

Le paramètre `with_pieces` permet de renvoyer, pour chaque token, le morceau de texte correspondant.

Le SDK permet également de récupérer le nombre de tokens via l'attribut `usage` de la réponse (prompt, complétion et total). Le nombre de tokens du prompt est un peu plus élevé qu'avec `/tokenize`, car le chat template ajoute ses balises autour du message.

In [29]:
prompt = '''Chaque modèle embarque son tokenizer. C'est important de mesurer le nombre de tokens envoyés et utilisés afin de ne pas dépasser la fenêtre de contexte.'''
r =  rq.post(f'{BASE_URL}/tokenize', json={'content': prompt, 'with_pieces': True})
tokens = r.json()['tokens']
print(f'{len(tokens)} tokens: {tokens}')

33 tokens: [{'id': 1106, 'piece': 'Ch'}, {'id': 19058, 'piece': 'aque'}, {'id': 79653, 'piece': ' modèle'}, {'id': 76857, 'piece': ' embar'}, {'id': 576, 'piece': 'que'}, {'id': 4292, 'piece': ' son'}, {'id': 44424, 'piece': ' tokenizer'}, {'id': 13, 'piece': '.'}, {'id': 351, 'piece': ' C'}, {'id': 16811, 'piece': "'est"}, {'id': 2894, 'piece': ' important'}, {'id': 401, 'piece': ' de'}, {'id': 10529, 'piece': ' mes'}, {'id': 7487, 'piece': 'urer'}, {'id': 501, 'piece': ' le'}, {'id': 12371, 'piece': ' nombre'}, {'id': 401, 'piece': ' de'}, {'id': 10885, 'piece': ' tokens'}, {'id': 57519, 'piece': ' envoy'}, {'id': 5229, 'piece': 'és'}, {'id': 1778, 'piece': ' et'}, {'id': 191257, 'piece': ' utilisés'}, {'id': 51511, 'piece': ' afin'}, {'id': 401, 'piece': ' de'}, {'id': 808, 'piece': ' ne'}, {'id': 6167, 'piece': ' pas'}, {'id': 212077, 'piece': ' dépass'}, {'id': 261, 'piece': 'er'}, {'id': 1147, 'piece': ' la'}, {'id': 195925, 'piece': ' fenêtre'}, {'id': 401, 'piece': ' de'}, {'id

In [30]:
r = client.chat.completions.create(
    model='local'
    , messages = [{'role': 'user', 'content': prompt}]
)
print(r.usage)

CompletionUsage(completion_tokens=1114, prompt_tokens=85, total_tokens=1199, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None))


## 6. Mesurer

C'est important de pouvoir mesurer la vitesse à laquelle vous traitez les prompts et générez des réponses.

Sans ces mesures, vous ne pouvez pas optimiser.

Llama-server renvoie ces mesures dans l'attribut `model_extra` de la réponse, sous la clé `timings`. Ce sont des champs propres à llama-server, hors du standard OpenAI, c'est pourquoi le SDK les range dans `model_extra`.

Une réponse se déroule en deux phases : le traitement du prompt (*prefill*), où tous les tokens d'entrée sont traités en parallèle, puis la génération (*decode*), où les tokens sont produits un par un.

| Mesure | Phase | Signification |
|--------|-------|---------------|
| `cache_n` | Prompt | Nombre de tokens du prompt retrouvés dans le KV cache, donc non recalculés. Avec un historique de conversation, le début du prompt est identique d'un tour à l'autre et llama-server le réutilise. |
| `prompt_n` | Prompt | Nombre de tokens du prompt effectivement traités (hors cache). |
| `prompt_ms` | Prompt | Temps total, en millisecondes, pour traiter le prompt. |
| `prompt_per_token_ms` | Prompt | Temps moyen par token de prompt (`prompt_ms / prompt_n`). |
| `prompt_per_second` | Prompt | Débit de traitement du prompt, en tokens par seconde. |
| `predicted_n` | Génération | Nombre de tokens générés dans la réponse. |
| `predicted_ms` | Génération | Temps total, en millisecondes, de la génération. |
| `predicted_per_token_ms` | Génération | Temps moyen pour générer un token (`predicted_ms / predicted_n`). |
| `predicted_per_second` | Génération | Débit de génération, en tokens par seconde. C'est la mesure la plus souvent citée (« tok/s »). |

Le débit du prompt est en général bien plus élevé que celui de la génération : le prefill exploite le parallélisme du GPU, alors que la génération est limitée par la bande passante mémoire, puisqu'il faut relire tous les poids du modèle pour chaque token produit.

In [34]:
r = client.chat.completions.create(
    model = 'local',
    messages = history
)

metrics = pl.from_dict(r.model_extra['timings'])
metrics

cache_n,prompt_n,prompt_ms,prompt_per_token_ms,prompt_per_second,predicted_n,predicted_ms,predicted_per_token_ms,predicted_per_second
i64,i64,f64,f64,f64,i64,f64,f64,f64
213,4,422.965,105.74125,9.457047,1,0.001,0.0,0.0
